In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Setup & models:

In [2]:
!pip install -q faiss-cpu

import torch, pandas as pd, numpy as np, faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline
from tqdm.auto import tqdm
import wandb
from kaggle_secrets import UserSecretsClient

wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))

DATA    = "/kaggle/input/competitions/smart-mcq-solver-challenge"
OPTIONS = ["A", "B", "C", "D", "E"]
DEV     = 0 if torch.cuda.is_available() else -1

RETRIEVE_K   = 10
CONTEXT_N    = 3
EMBED_MODEL  = "sentence-transformers/all-mpnet-base-v2"
READER_MODEL = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"

train = pd.read_csv(f"{DATA}/train.csv")
test  = pd.read_csv(f"{DATA}/test.csv")

kb = [str(row[row["answer"]]) for _, row in train.iterrows()]

embedder = SentenceTransformer(EMBED_MODEL)
kb_emb = embedder.encode(kb, normalize_embeddings=True, show_progress_bar=True).astype("float32")
index = faiss.IndexFlatIP(kb_emb.shape[1])
index.add(kb_emb)

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
zs = pipeline("zero-shot-classification", model=READER_MODEL, device=DEV)
print("Improved RAG ready. KB docs:", index.ntotal)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 75.6 MB/s eta 0:00:00:00:0100:01


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ishankgpt02 (ishankgpt02-na) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

Improved RAG ready. KB docs: 2000


# RAG function

In [3]:
def retrieve_indices(prompt, k, exclude=None):
    q = embedder.encode([str(prompt)], normalize_embeddings=True).astype("float32")
    _, I = index.search(q, k + (1 if exclude is not None else 0))
    return [int(i) for i in I[0] if i != exclude][:k]

def rag_top3(prompt, options_texts, exclude=None):
    cand = retrieve_indices(prompt, RETRIEVE_K, exclude)
    ce = cross_encoder.predict([[str(prompt), kb[i]] for i in cand])
    best = [cand[j] for j in np.argsort(ce)[::-1][:CONTEXT_N]]
    context = " ".join(kb[i] for i in best)
    rag = f"Context: {context} Question: {prompt}"
    res = zs(rag, candidate_labels=options_texts)
    return [OPTIONS[options_texts.index(lab)] for lab in res["labels"]]

def map3_row(ranked, truth):
    for i, l in enumerate(ranked[:3]):
        if l == truth:
            return 1.0 / (i + 1)
    return 0.0

# Context-sweep

In [4]:
run = wandb.init(entity="ishankgpt02-na", project="23f1002033-t22026",
                 name="rag-v2-context-sweep")

for n in [1, 2, 3, 5]:
    maps = []
    for i in range(100):
        r = train.iloc[i]
        opts = [str(r[o]) for o in OPTIONS]
        cand = retrieve_indices(str(r["prompt"]), RETRIEVE_K, exclude=i)
        ce = cross_encoder.predict([[str(r["prompt"]), kb[j]] for j in cand])
        best = [cand[j] for j in np.argsort(ce)[::-1][:n]]
        context = " ".join(kb[j] for j in best)
        res = zs(f"Context: {context} Question: {r['prompt']}", candidate_labels=opts)
        ranked = [OPTIONS[opts.index(lab)] for lab in res["labels"]]
        maps.append(map3_row(ranked, r["answer"]))
    score = float(np.mean(maps))
    print(f"context_n={n}: MAP@3={score:.4f}")
    wandb.log({"context_n": n, "map@3": score})

run.finish()
print("Context-sweep logged to W&B")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


context_n=1: MAP@3=0.8917
context_n=2: MAP@3=0.9233
context_n=3: MAP@3=0.9333
context_n=5: MAP@3=0.9317


context_n,▁▃▅█
map@3,▁▆██
context_n,5
map@3,0.93167


Context-sweep logged to W&B


# Test submission

In [5]:
preds = []
for i in tqdm(range(len(test)), desc="test submission"):
    r = test.iloc[i]
    opts = [str(r[o]) for o in OPTIONS]
    ranked = rag_top3(str(r["prompt"]), opts)      
    preds.append(" ".join(ranked[:3]))

submission = pd.DataFrame({"ID": test["id"], "Prediction": preds})
submission.to_csv("submission.csv", index=False)
print("submission.csv saved:", submission.shape)

test submission:   0%|          | 0/500 [00:00<?, ?it/s]

submission.csv saved: (500, 2)


# W&B logging

In [6]:
run = wandb.init(entity="ishankgpt02-na", project="23f1002033-t22026",
                 name="rag-v2-improved",
                 config={"retriever": EMBED_MODEL, "index": "cosine (IP)",
                         "reranker": "ms-marco-MiniLM-L-6-v2",
                         "reader": READER_MODEL,
                         "retrieve_k": RETRIEVE_K, "context_n": CONTEXT_N})
run.finish()
print("rag-v2-improved logged to W&B")

rag-v2-improved logged to W&B
